[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Crew/transport-networks-lab/blob/main/Notebooks/Practica_guiada4_ContOSMNx194.ipynb)


#  Grafos y redes de transporte

##  OSMnx Parte II
### Topología de une red de calles
#### Algunos conceptos básicos

Los nodos de OpenStreetMap no sólo contemplan intersecciones. También suelen incluir puntos que no cumplen ninguna función en el sentido estricto de la teoría de grafos. Como por ejemplo, puntos en curvaturas de calles. Al decir que no cumplen ninguna función, no estamos diciendo otra cosa que no son nodos y como tales no forman parte de nuestra red. Sí corresponden a algún segmento de un eje, pero que necesita ser simplificado.

In [ ]:
#%%capture
!pip install osmnx==1.9.4

In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
ox.__version__
print(ox.__version__)
ox.settings.use_cache=True

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Simplificando la red

In [ ]:
ox.graph_from_point?

In [ ]:
# creamos una red a partir de un set de coordenadas
referencia = (-34.50944,-58.58610)
G = ox.graph_from_point(referencia, network_type='walk', dist=500, simplify=False)

In [ ]:
# el método "simplify_graph" elimina los nodos que no son un endpoint
endpoint_attrs = ['highway']
nc = ['r' if ox.simplification._is_endpoint(G, node, endpoint_attrs) else 'y' for node in G.nodes()]

In [ ]:
fig, ax = ox.plot_graph(G, bgcolor='black', node_color=nc)

Detengámonos dos segundos a ver qué pasó acá. En primer lugar, el atributo `nodo` de nuestra red parece ser un iterable. Veamos qué tipo de objeto es ...

In [ ]:
# ya vemos que los nodos se definen a partir de clases de la librería netwrokx
type(G.nodes())

En nuestro caso, contamos con ...

In [ ]:
len(G.nodes())

In [ ]:
# y ...
len(G.edges())

Vemoas qué sucede si accedemos al primer nodo de nuestra lista y lo evaluamos con el método `_is_endpoint`

In [ ]:
primer_nodo = list(G.nodes())[0]
ox.simplification._is_endpoint(G, primer_nodo, endpoint_attrs)

Esto nos devuelve un booleano, lo cual nos permite apelar a una lista por comprensión para obtener los colores que se le va a asignar a cada eje.

Vamos a remover los puntos amarillos, ya que no son nodos en si mismos.

In [ ]:
# ahora sí, simplificamos la red y la ploteamos. Vemos que los nodos amarillos desaparecieron.
# Nos quedamos solamente con los end points
G = ox.simplify_graph(G)
fig, ax = ox.plot_graph(G, node_color='r')

In [ ]:
len(G.nodes())

### Ejercitación:

Cómo harían para disponer los nodos y los ejes de manera relacional? Es decir, saber que los pares de ejes corresponden a tal o cual nodos?

In [ ]:
#@title **Solucion**
def arma_ejes(x):
    for i in G.edges():
        if x in i:
            return i
        else:
            pass

ejes = pd.Series(list(G.nodes())).map(arma_ejes)
#ejes.head()

grafo_df = pd.DataFrame({'nodos':list(G.nodes()), 'ejes':ejes})
grafo_dict = dict(zip(grafo_df.nodos, grafo_df.ejes))

# así podríamos acceder a un par de ejes por alguno de sus nodos
grafo_dict[333739864]

# 2. Características de la red (ejes)

De todas manera, OSMNx cuenta con sus propios métodos para estructurar una red con nodos y arcos sin tener que recurrir a pandas o estructuras de python nativo. En esta sección vamos a ver cómo caracterizar una red en función de sus atributos, tanto trabajando con la clase `graph` de `networkx` como reestructurando el grafo como un geodataframe.

In [ ]:
# primero, recordemos que estamos trabajando con un grafo dirigido de ambos sentidos
type(G)

## 2.1. Largo de calles

Algo muy útil de OSMNx es que te permite calcular colores por atributos. En este caso, veamos cómo funciona con el largo de las calles (o ejes de nuestro grafo)...

In [ ]:
# este método pertenece al módulo plot
largo_ejes = ox.plot.get_edge_colors_by_attr(G, attr='length', cmap='YlOrBr')
fig, ax = ox.plot_graph(G, bgcolor='black', node_color='w', node_edgecolor='k', node_size=50,
                        edge_color=largo_ejes, edge_linewidth=3)

Podemos ver que este método nos devuelve un set ordenado de datos de tipo ...

In [ ]:
# Serie, de pandas
type(largo_ejes)

In [ ]:
largo_ejes

Esto representa a cada uno de los nodos con sus respectivas conexiones. Así quedan armados los ejes, a los cuales podemos acceder indexando por el id de los extremos.

In [ ]:
# acá el código con el color asigando
largo_ejes[333739864][333739879][0]

El color representado por la tupla (0.996078431372549, 0.8701730103806228, 0.5259976931949251, 1.0) está en formato RGBA, donde cada valor corresponde a:

* R (Rojo): 0.996 (aproximadamente 1.0, es decir, un valor muy alto de rojo),
* G (Verde): 0.870 (un valor intermedio de verde),
* B (Azul): 0.526 (un valor intermedio de azul),
* A (Alfa): 1.0 (opacidad completa, es decir, el color no es transparente).

## 2.2. Calles de sentido único

In [ ]:
from google.colab import drive
drive.mount('/drive/')

In [ ]:
path = '/drive/MyDrive/Técnicas y Análisis de datos del Transporte/Data/renabap.geojson'
barrios = gpd.read_file(path)

# nos quedamos con los barrios de San Martin
barrios_sm = barrios.loc[(barrios.provincia == 'Buenos Aires') & (barrios.departamen == 'General San Martín')]

In [ ]:
# y ahora con algunos de jose leon suarez
barrios_ls = barrios_sm.loc[(barrios_sm.nombre_bar == 'Villa Hidalgo')|
                            (barrios_sm.nombre_bar == 'La Carcova')].copy()

In [ ]:
# La Carcova
lc = barrios_ls['geometry'].iloc[1]

In [ ]:
G2 = ox.graph_from_polygon(lc, network_type='drive_service')

In [ ]:
fig, ax = ox.plot_graph(G2, bgcolor='white', node_color='blue', node_size=35)

In [ ]:
# veamos ahora cuales son las calles de sentido unico
colores = ['red' if data['oneway'] else 'grey' for u, v, key, data in G2.edges(keys=True, data=True)]
fig, ax = ox.plot_graph(G2, node_size=0, bgcolor='white', edge_color=colores, edge_linewidth=2.5, edge_alpha=0.7)

Antes, vimos que el parametro `keys` nos devolvía los nodos de un eje, ahora podemos apreciar que con `data` conseguimos información adicional, como el nombre o la direccionalidad de nuestra calle.

In [ ]:
list(G2.edges(keys=True, data=True))[3:5]

# 3. Métricas de una red

## 3.1. Estadísticas de base

OSMNx también cuenta con el método `basic_stats`. Este nos da un pantallazo acerca de la topología de la red, con métricas como la cantidad de nodos y ejes, el largo total de las calles, etc.

In [ ]:
# calculamos métricas de base
stats = ox.basic_stats(G)

In [ ]:
# que vemos se almacenan en un diccionario
stats.keys()

In [ ]:
stats

In [ ]:
# al que podemos acceder como en cualquier diccionario, por ejemplo el largo promedio de las calles
stats['street_length_avg']

### Resúmen de métricas del módulo [stats](https://github.com/gboeing/osmnx/blob/master/osmnx/stats.py) se puede encontrar en la siguiente [documentación](https://osmnx.readthedocs.io/en/stable/user-reference.html#module-osmnx.stats)

In [ ]:
# calculemos el área de nuestra red
lugar = 'Villa Hidalgo, José León Suárez, Partido de General San Martín, Buenos Aires'
gdf = ox.geocode_to_gdf(lugar)
area = ox.project_gdf(gdf).unary_union.area

In [ ]:
# recuerdan que el método geocode_to_gdf nos devolvía un gdf con el polígono de nuestra consulta?
gdf

In [ ]:
area

**Con el parametro area incluimos métricas de densidad:**

In [ ]:
# volvemos a calcular nuestros estadísticos de base
stats = ox.basic_stats(G, area=area)

In [ ]:
# Nos enfocamos en los de densidad
for k,v in stats.items():
    if 'density' in k:
        print(k,':', stats[k])

Para interpretear estos resultados debemos tener en cuenta cada una de las keys de nuestro diccionario stats y el area en km2 cubierta por nuestra red.

In [ ]:
stats['n']/(area/1e6)  # la cantidad de nodos por km2 en el area de nuestra red

In [ ]:
stats['edge_length_total']/(area/1e6)  # longitud total de los ejes en km por km2 en el area de nuestra red

## 3.2. Otras estadísticas

In [ ]:
dir(ox.stats)

In [ ]:
extended_stats = ox.stats.count_streets_per_node(G)

In [ ]:
extended_stats

## 3.2.1. Betweenness centrality

In [ ]:
# reproyectar nuestro grafo
G_proj = ox.project_graph(G)

In [ ]:
nx.betweenness_centrality?

In [ ]:
# Calculamos la centralidad de intermediacion para los nodos
node_betweenness = nx.betweenness_centrality(G_proj, weight='length', normalized=True)

# E identificamos los 10 nodos con mayor centralidad
sorted_node_betweenness = sorted(node_betweenness.items(), key=lambda item: item[1], reverse=True)
for node, centrality in sorted_node_betweenness[:10]:
    print(f"Node: {node}, Betweenness Centrality: {centrality}")


In [ ]:
# Hacemos lo mismo para los ejes
edge_betweenness = nx.edge_betweenness_centrality(G_proj, weight='length', normalized=True)

sorted_edge_betweenness = sorted(edge_betweenness.items(), key=lambda item: item[1], reverse=True)
for edge, centrality in sorted_edge_betweenness[:10]:
    print(f"Edge: {edge}, Betweenness Centrality: {centrality}")


In [ ]:
# Calculamos centralidad de intermediacion
node_betweenness = nx.betweenness_centrality(G_proj, weight='length', normalized=True)

# Colores y tamaños segn el grado de centralidad
nc = [node_betweenness[node] for node in G_proj.nodes()]

see_top_nodes = False
if see_top_nodes:
  ns = [50 * node_betweenness[node] for node in G_proj.nodes()]
else:
  ns = [max(10, 1000 * node_betweenness[node]) for node in G_proj.nodes()]


# Graficamos
fig, ax = ox.plot_graph(
    G_proj,
    node_color=nc,
    node_size=ns,
    node_zorder=2,
    edge_linewidth=0.5,
    edge_color='#999999',
    show=False,
    close=False
)

# Creamos un colormap con el minimo y el maximo de centralidad como limites
sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=min(nc), vmax=max(nc)))
sm._A = []

# Eje adicional para el colorbar
cax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Betweenness Centrality')

plt.show()


In [ ]:
# Calculamos la centralidad de intermediacion para los ejes
edge_betweenness = nx.edge_betweenness_centrality(G_proj, weight='length', normalized=True)

# Placeholders para el color y el width del eje segun el nivel de centralidad
ec = []
ew = []

# iteramos sobre un multigrafo (ejes paralelos)
for u, v, key, data in G_proj.edges(keys=True, data=True):
    edge = (u, v, key)
    betweenness = edge_betweenness[edge]
    ec.append(betweenness)
    ew.append(max(0.5, 30 * betweenness))

# Normalizamos los colores de los ejes
norm = plt.Normalize(vmin=min(ec), vmax=max(ec))
cmap = plt.colormaps.get_cmap('viridis')
ec = [cmap(norm(val)) for val in ec]

fig, ax = ox.plot_graph(
    G_proj,
    node_size=0,  # el valor del nodo es cero para focalizarnos en los ejes
    node_zorder=2,
    edge_color=ec,
    edge_linewidth=ew,
    show=False,
    close=False,
    bgcolor='white'
)

# Colorbar para el betweenness centrality de los ejes
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []

# Nuevo eje para el colorbar
cax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Edge Betweenness Centrality')

plt.show();


# 4. Shortest path routing

Por último, vamos a tratar de ver cómo el concepto de `shortest_path` que vimos en clases anteriores se implementan en OSMNx. Existen varias formas de encontrar la ruta más cercana entre dos puntos, una muy común es ponderar los ejes por la distancia o el tiempo de viaje.

In [ ]:
from pyproj import Transformer
print(G.graph['crs'])

In [ ]:
len(list(G_proj.nodes()))

In [ ]:
# elegimos un punto de referencia para nuestro nodo de origen
referencia = (-34.505754, -58.586659)

# convertimos las coordenadas de referencia a UTM
transformer = Transformer.from_crs("epsg:4326", "epsg:32721")
x, y = transformer.transform(referencia[0], referencia[1])
print(f"Coordenadas convertidas del punto de referencia: X: {x}, Y: {y}")

origin_node = ox.distance.nearest_nodes(G_proj, X=x, Y=y)
destination_node = list(G_proj.nodes())[2]

print(f"Nodo origen: {origin_node}")
print(f"Nodo destino: {destination_node}")

In [ ]:
nx.shortest_path?

In [ ]:
# ahora calculamos el shortest path entre ambos y ploteamos
route = nx.shortest_path(G, origin_node, destination_node)

fig, ax = ox.plot_graph_route(G, route)

# Ejercitación

Descargar la red de calles de:

**a)** Dos asentamientos informales

**b)** Un asentamiento informal y otro de traza regular

... y compararlas considerando los principales aspectos topológicos de ambas ¿Cómo afectan estos a la movilidad dentro de la red?

*Nota*
Entre los distintos métodos que se revisaron para descargar una red, elegir aquel que garantice la mayor comparabilidad. Asimismo, elegir el conjunto de métricas que mejor describan la movilidad dentro del contexto analizado.